# RFDC V4 Pump + Pi/2 + Probe — TTL-Triggered Repeated Production Loop

Repeated-shot qualification of the same production pump + Pi/2 + probe sequence and TX/RX
gate discipline as `RFDC_V4_Pump_Pi2_Probe_Loop_Qualification(1)(4).ipynb`, except every shot
is started by a **hardware TTL edge on `trig_in`** instead of a software write to
`SEQ_ENABLE`. Run `RFDC_V4_Pump_Pi2_Probe_TTL_LongCapture.ipynb` first and confirm its single
shot and plots look right before running this loop.

## Required physical setup

Same jumper as the single-shot notebook: `sync_out` -> `trig_in`, i.e. **AW15 -> AW16** on the
Pmod+ header (confirmed against `base.xdc`). This is the bring-up stand-in for the DG4000, not
the final wiring.

## Read this before trusting a PASS here

1. **Bitstream check.** This notebook only exercises the RTL that's actually loaded. If the
   `AXI_Pulse_Sequencer` instance in the loaded `.bit` still has the old 6-bit address-width
   customization instead of the TTL build's 7-bit width, every TTL register write below is a
   silent no-op and `run_shot()` hangs on its first `RUN_DONE` wait with no other symptom.
2. **`TTL_EDGE_COUNT` resets every shot.** `prepare_shot()` below issues a manual `SEQ_RESET`
   before each shot, matching the original notebook's per-shot discipline. In this TTL build,
   that same reset also zeroes `TTL_ARM`/`RUNNING`/`RUN_DONE`/**`TTL_EDGE_COUNT`** (see
   `AXI_Pulse_Sequencer.vhd`'s `TTL_RUN_FSM`, `reset_strobe` branch) -- it is not scoped to
   just row/counter state the way it is in plain internal-trigger mode. So each shot's
   correctness check is `edge_count == 1` (a fresh count from a known-zero baseline), not
   "incremented by one from whatever it was" the way the isolated loopback self-test checks
   it (that notebook never resets `seq` between iterations, so it accumulates instead).
3. **Guard-row gap shortened.** Row 22 (mask=0 guard) uses `TTL_GUARD_GAP_S` instead of the
   original notebook's 100 ms, because `RUN_DONE` can't assert until every row's
   `row_advance` has fired, including the guard row's. This doesn't touch the timing of the
   pump/Pi2/probe rows (0-21) or `CAPTURE_S` -- see the single-shot notebook's markdown for
   the full reasoning.
4. **This still doesn't add capture-gate overflow protection.** TTL mode only changes what
   starts row 0. The confirm-idle -> clear -> rearm-DMA -> **then** `TTL_ARM` ordering in
   `prepare_shot()`/`arm_shot()`/`run_shot()` below is still 100% what's preventing the
   shot-268-style overflow race, exactly as it was for `SEQ_ENABLE` in the original loop.

## Ethernet offload (added in this version)

Each shot's RX capture (ADC A + ADC B, ~14 MiB total for the default 30 ms `CAPTURE_S`) is
streamed to a PC over the RJ45 Ethernet port immediately after that shot's hardware is
confirmed idle, using a small length-prefixed TCP protocol (see the network-helpers cell).
The companion PC-side receiver/plotting notebook is
`RFDC_PC_Ethernet_Receiver.ipynb` -- **start that notebook first** (it listens and waits for
this board to connect) before running this one.

This version also times, for every shot, the gap between "this shot's hardware finished and
its data is valid" and "the next shot is armed and its own Ethernet transfer has been ACKed
by the PC" -- i.e. the dead time an external trigger source (DG4000 or otherwise) would need
to respect so it never fires while the board is still mid-transfer or mid-rearm. See the
summary cell at the end for the measured max/mean and what it implies for `INTER_SHOT_PAUSE_S`
and for programming a future external trigger's repetition rate.


In [1]:
import time
import numpy as np
import xrfclk
import xrfdc
from pynq import Overlay, allocate

# ================= USER CONFIG =================
# Confirm this is the rebuilt bitstream with the TTL-updated AXI_Pulse_Sequencer
# (C_S_AXI_ADDR_WIDTH=7, registers 0x34-0x44) -- NOT necessarily the same reset8.bit used by
# the software-timed notebooks.
BITFILE = "./reset10.bit"

AXIS_BEAT_HZ = 15.36e6
SAMPLES_PER_BEAT = 8
SEQ_CLK_HZ = 99_999_985.0

DAC_A_NCO_MHZ = 10.0
DAC_B_NCO_MHZ = 0.2

PUMP_REPS = 10
PUMP_PULSE_S = 1e-3
PUMP_ROW_SPACING_S = 1.005e-3
DAC_A_CHIRP_START_MHZ = 5.0
DAC_A_CHIRP_END_MHZ = 10.0
DAC_A_SINE_MHZ = 15.0

PI2_PULSE_S = 1e-3
PI2_GAP_S = 1e-3

PROBE_PULSE_S = 10e-6
PROBE_GAP_S = 1e-6
DAC_A_PROBE_MHZ = [10.0, 11.0, 12.0, 13.0, 14.0]

CAPTURE_S = 30e-3
DMA_ARM_SETTLE_S = 5e-3
LOOP_COUNT = 2000
INTER_SHOT_PAUSE_S = 2e-3
DISCARD_RX_DATA = True
FAIL_FAST = True
AMPLITUDE = 32760

POST_SEQUENCE_MARGIN_S = 2e-3
OVERVIEW_POINTS = 12000

# ================= ETHERNET OFFLOAD CONFIG =================
# PC running RFDC_PC_Ethernet_Receiver.ipynb, reachable over the RJ45 link.
PC_HOST = "192.168.3.137"      # <-- set this to the PC's IP address on the RFSoC's Ethernet link
PC_PORT = 5001
# Generous on purpose: this only bounds how long we wait for a *stuck* transfer before
# raising, it does not add real per-shot delay when things are healthy (see markdown/summary
# cell for the actually-measured per-shot Ethernet time).
ETH_TIMEOUT_S = 15.0
# Informational only (used for the printed estimate below); not enforced anywhere.
ETHERNET_LINK_MBPS_ESTIMATE = 190

# TTL-mode-specific: see markdown above.
TTL_GUARD_GAP_S = POST_SEQUENCE_MARGIN_S

EXPECTED_IP_PATHS = {
    "sequencer": "radio/AXI_Pulse_Sequencer_0",
    "tx_gate_b": "radio/AXI_TX_Multi_Gate_0",
    "tx_gate_a": "radio/AXI_TX_Multi_Gate_1",
    "cap_gate_b": "radio/receiver/channel_20/AXI_Capture_Gate_0",
    "cap_gate_a": "radio/receiver/channel_21/AXI_Capture_Gate_0",
    "rx_dma_b": "radio/receiver/channel_20/axi_dma_real",
    "rx_dma_a": "radio/receiver/channel_21/axi_dma_real",
    "tx_dma_b": "radio/axi_dma_dac_0",
    "tx_dma_a": "radio/axi_dma_dac_1",
}

print("Loading overlay...")
base = Overlay(BITFILE, download=True)
print("Overlay loaded.")


Loading overlay...


Overlay loaded.


In [2]:
def get_by_path(root, path):
    obj = root
    for part in path.split('/'):
        obj = getattr(obj, part)
    return obj

missing = [p for p in EXPECTED_IP_PATHS.values() if p not in base.ip_dict]
if missing:
    print("Missing expected HWH paths:")
    for p in missing:
        print("  ", p)
    raise KeyError("The loaded .hwh does not match the validated A/B topology.")

seq = get_by_path(base, EXPECTED_IP_PATHS['sequencer'])
tx_b = get_by_path(base, EXPECTED_IP_PATHS['tx_gate_b'])
tx_a = get_by_path(base, EXPECTED_IP_PATHS['tx_gate_a'])
cap_b = get_by_path(base, EXPECTED_IP_PATHS['cap_gate_b'])
cap_a = get_by_path(base, EXPECTED_IP_PATHS['cap_gate_a'])
dma_rb = get_by_path(base, EXPECTED_IP_PATHS['rx_dma_b'])
dma_ra = get_by_path(base, EXPECTED_IP_PATHS['rx_dma_a'])
dma_tb = get_by_path(base, EXPECTED_IP_PATHS['tx_dma_b'])
dma_ta = get_by_path(base, EXPECTED_IP_PATHS['tx_dma_a'])
print('IP resolved.')


IP resolved.


In [3]:
# ================= REGISTER MAP =================
SEQ_ROW_SEL=0x00; SEQ_MASK=0x04; SEQ_GAP=0x08
SEQ_DUR0=0x0C; SEQ_DUR1=0x10; SEQ_DUR2=0x14; SEQ_DUR3=0x18
SEQ_COMMIT=0x1C; SEQ_TABLE_LEN=0x20; SEQ_ENABLE=0x24
SEQ_RESET=0x28; SEQ_ROW_PTR=0x2C; SEQ_CDC_OVERRUN=0x30

# New TTL registers (see AXI_Pulse_Sequencer.vhd header, 0x34-0x44).
SEQ_TTL_ARM=0x34
SEQ_TTL_MODE=0x38
SEQ_TTL_STATUS=0x3C
SEQ_TTL_STATUS_CLEAR=0x40
SEQ_TTL_EDGE_COUNT=0x44

TTL_ARMED=1<<0; TTL_RUNNING=1<<1; TTL_RUN_DONE=1<<2

TXM_SEG_SEL=0x00; TXM_ACTIVE_LEN=0x04; TXM_GAP_LEN=0x08; TXM_COMMIT=0x0C
TXM_NUM_SEGMENTS=0x10; TXM_STATUS=0x1C
TX_BUSY=1<<0; TX_OVERRUN=1<<1

CAP_LENGTH = 0x00
CAP_STATUS = 0x08
CAP_CLEAR = 0x0C
CAP_BUSY = 1 << 0
CAP_OVERFLOW = 1 << 1

LANE_TX_B=0; LANE_TX_A=1; LANE_RX_B=2; LANE_RX_A=3
MASK_ALL=0xF

def beats_for(seconds):
    return max(1, int(round(seconds * AXIS_BEAT_HZ)))

def samples_for(beats):
    return int(beats) * SAMPLES_PER_BEAT

def seq_cycles_for(seconds):
    return max(1, int(round(seconds * SEQ_CLK_HZ)))

def pack_iq(i, q):
    if len(i) != len(q):
        raise ValueError('I/Q length mismatch')
    out = np.empty(2*len(i), dtype=np.int16)
    out[0::2] = i
    out[1::2] = q
    return out

def tone_iq(n, f_hz, fs=122.88e6, phase_deg=0.0):
    t = np.arange(int(n), dtype=np.float64) / fs
    ph = 2*np.pi*f_hz*t + np.deg2rad(phase_deg)
    return (np.round(AMPLITUDE*np.cos(ph)).astype(np.int16),
            np.round(AMPLITUDE*np.sin(ph)).astype(np.int16))

def chirp_iq(n, f0_hz, f1_hz, duration_s, fs=122.88e6):
    t = np.arange(int(n), dtype=np.float64) / fs
    k = (f1_hz-f0_hz)/duration_s
    ph = 2*np.pi*(f0_hz*t + 0.5*k*t*t)
    return (np.round(AMPLITUDE*np.cos(ph)).astype(np.int16),
            np.round(AMPLITUDE*np.sin(ph)).astype(np.int16))

def ttl_status_val(s):
    return int(s.mmio.read(SEQ_TTL_STATUS))

def ttl_status_str(v):
    flags=[]
    if v & TTL_ARMED:    flags.append("ARMED")
    if v & TTL_RUNNING:  flags.append("RUNNING")
    if v & TTL_RUN_DONE: flags.append("RUN_DONE")
    return "|".join(flags) if flags else "idle"

def ttl_arm(s):
    s.mmio.write(SEQ_TTL_ARM, 1)

def ttl_edge_count(s):
    return int(s.mmio.read(SEQ_TTL_EDGE_COUNT))

print('Helpers ready.')


Helpers ready.


In [4]:
# ================= RFDC STATE =================
rfdc = get_by_path(base, 'radio/rfdc')
dac_b = rfdc.dac_tiles[0].blocks[0]
dac_a = rfdc.dac_tiles[2].blocks[0]
adc_b = rfdc.adc_tiles[2].blocks[0]
adc_a = rfdc.adc_tiles[2].blocks[1]

print('DAC B mixer before:', dac_b.MixerSettings)
print('DAC A mixer before:', dac_a.MixerSettings)

for dac, freq in ((dac_b,DAC_B_NCO_MHZ),(dac_a,DAC_A_NCO_MHZ)):
    ms = dac.MixerSettings
    ms['Freq'] = float(freq)
    ms['PhaseOffset'] = 0.0
    ms['EventSource'] = 2
    dac.MixerSettings = ms
    dac.UpdateEvent(xrfdc.EVENT_MIXER)

print('DAC B mixer after:', dac_b.MixerSettings)
print('DAC A mixer after:', dac_a.MixerSettings)
print('ADC B mixer:', adc_b.MixerSettings)
print('ADC A mixer:', adc_a.MixerSettings)

for dac,label,freq in ((dac_b,'DAC B',DAC_B_NCO_MHZ),(dac_a,'DAC A',DAC_A_NCO_MHZ)):
    ms=dac.MixerSettings
    assert abs(float(ms['Freq'])-freq) < 1e-6
    assert float(ms['PhaseOffset']) == 0.0
    assert int(ms['EventSource']) == 2
print('PASS: explicit DAC mixer state matches this notebook.')


DAC B mixer before: {'Freq': 0.0, 'PhaseOffset': 0.0, 'EventSource': 2, 'CoarseMixFreq': 16, 'MixerMode': 2, 'FineMixerScale': 0, 'MixerType': 2}
DAC A mixer before: {'Freq': 0.0, 'PhaseOffset': 0.0, 'EventSource': 2, 'CoarseMixFreq': 16, 'MixerMode': 2, 'FineMixerScale': 0, 'MixerType': 2}
DAC B mixer after: {'Freq': 0.19999999998835846, 'PhaseOffset': 0.0, 'EventSource': 2, 'CoarseMixFreq': 0, 'MixerMode': 2, 'FineMixerScale': 0, 'MixerType': 2}
DAC A mixer after: {'Freq': 9.999999999994179, 'PhaseOffset': 0.0, 'EventSource': 2, 'CoarseMixFreq': 0, 'MixerMode': 2, 'FineMixerScale': 0, 'MixerType': 2}
ADC B mixer: {'Freq': 0.0, 'PhaseOffset': 0.0, 'EventSource': 2, 'CoarseMixFreq': 16, 'MixerMode': 4, 'FineMixerScale': 0, 'MixerType': 1}
ADC A mixer: {'Freq': 0.0, 'PhaseOffset': 0.0, 'EventSource': 2, 'CoarseMixFreq': 16, 'MixerMode': 4, 'FineMixerScale': 0, 'MixerType': 1}
PASS: explicit DAC mixer state matches this notebook.


In [5]:
# ================= DMA / CAPTURE CAPACITY SELF-CHECK =================
capture_beats = beats_for(CAPTURE_S)
capture_samples = samples_for(capture_beats)
capture_bytes = capture_samples * 2
print(f'Capture duration: {capture_beats/AXIS_BEAT_HZ*1e3:.6f} ms')
print(f'Capture samples/channel: {capture_samples:,}')
print(f'Capture bytes/channel: {capture_bytes:,} ({capture_bytes/1024/1024:.2f} MiB)')

if CAPTURE_S > 260e-3 + 1e-12:
    raise ValueError('This notebook intentionally limits capture to 260 ms by default.')

DMA_MAX_BYTES = (1 << 26) - 1
print("DMA configured buffer-length limit:", DMA_MAX_BYTES, "bytes")
print("Capture bytes/channel:", capture_bytes)

if capture_bytes > DMA_MAX_BYTES:
    raise RuntimeError(f"Capture requires {capture_bytes:,} bytes, but the configured 26-bit DMA length limit is {DMA_MAX_BYTES:,} bytes")
print("PASS: capture fits the configured 26-bit DMA transfer length.")

pump_beats = beats_for(PUMP_PULSE_S)
probe_beats = beats_for(PROBE_PULSE_S)
pi2_beats = beats_for(PI2_PULSE_S)
pump_samples = samples_for(pump_beats)
probe_samples = samples_for(probe_beats)
pi2_samples = samples_for(pi2_beats)

expected_tx_a_i16 = 2*(20*pump_samples + 5*probe_samples)
expected_tx_b_i16 = 2*(2*pi2_samples)
print(f'Expected TX-A payload: {expected_tx_a_i16:,} int16 = {expected_tx_a_i16*2/1024/1024:.2f} MiB')
print(f'Expected TX-B payload: {expected_tx_b_i16:,} int16 = {expected_tx_b_i16*2/1024/1024:.2f} MiB')

tx_a_bytes = expected_tx_a_i16 * 2
tx_b_bytes = expected_tx_b_i16 * 2
if tx_a_bytes > DMA_MAX_BYTES:
    raise RuntimeError(f"TX-A payload {tx_a_bytes:,} bytes exceeds {DMA_MAX_BYTES:,}-byte DMA limit")
if tx_b_bytes > DMA_MAX_BYTES:
    raise RuntimeError(f"TX-B payload {tx_b_bytes:,} bytes exceeds {DMA_MAX_BYTES:,}-byte DMA limit")
print("PASS: both TX payloads fit the configured DMA transfer length.")

if capture_bytes >= 2**26:
    raise RuntimeError('RX capture exceeds the 26-bit simple-DMA length field.')
print('PASS: DMA transfer sizes are within the configured simple-DMA limits.')

# ---- Informational: expected per-shot Ethernet transfer time at the configured link estimate.
# This is payload-only (no TCP/IP framing or protocol overhead); see the markdown at the top
# of this notebook / the accompanying chat writeup for the full derivation.
_total_rx_bytes_per_shot = 2 * capture_bytes
_est_eth_s = _total_rx_bytes_per_shot / (ETHERNET_LINK_MBPS_ESTIMATE * 1e6 / 8)
print(f'Total RX payload per shot (A+B): {_total_rx_bytes_per_shot:,} bytes '
      f'({_total_rx_bytes_per_shot/1024/1024:.2f} MiB)')
print(f'Estimated per-shot Ethernet transfer time at {ETHERNET_LINK_MBPS_ESTIMATE} Mbps: '
      f'{_est_eth_s*1e3:.0f} ms (payload only; actual will run a bit higher with TCP/IP '
      f'overhead and the PC-side write+ACK -- the loop below measures the real number).')


Capture duration: 30.000000 ms
Capture samples/channel: 3,686,400
Capture bytes/channel: 7,372,800 (7.03 MiB)
DMA configured buffer-length limit: 67108863 bytes
Capture bytes/channel: 7372800
PASS: capture fits the configured 26-bit DMA transfer length.
Expected TX-A payload: 4,927,520 int16 = 9.40 MiB
Expected TX-B payload: 491,520 int16 = 0.94 MiB
PASS: both TX payloads fit the configured DMA transfer length.
PASS: DMA transfer sizes are within the configured simple-DMA limits.
Total RX payload per shot (A+B): 14,745,600 bytes (14.06 MiB)
Estimated per-shot Ethernet transfer time at 190 Mbps: 621 ms (payload only; actual will run a bit higher with TCP/IP overhead and the PC-side write+ACK -- the loop below measures the real number).


In [6]:
# ================= WAVEFORM BUILD =================
fs_a=122.88e6; fs_b=122.88e6

chirp_n=pump_samples
chirp_i,chirp_q = chirp_iq(chirp_n,
    (DAC_A_CHIRP_START_MHZ-DAC_A_NCO_MHZ)*1e6,
    (DAC_A_CHIRP_END_MHZ-DAC_A_NCO_MHZ)*1e6,
    PUMP_PULSE_S, fs_a)
sine_i,sine_q = tone_iq(pump_samples,(DAC_A_SINE_MHZ-DAC_A_NCO_MHZ)*1e6,fs=fs_a)
probe_parts_i=[]; probe_parts_q=[]
for f_abs in DAC_A_PROBE_MHZ:
    ii,qq=tone_iq(probe_samples,(f_abs-DAC_A_NCO_MHZ)*1e6,fs=fs_a)
    probe_parts_i.append(ii); probe_parts_q.append(qq)

pump_i = np.concatenate(([np.concatenate([chirp_i,sine_i])]*PUMP_REPS))
pump_q = np.concatenate(([np.concatenate([chirp_q,sine_q])]*PUMP_REPS))
probe_i=np.concatenate(probe_parts_i); probe_q=np.concatenate(probe_parts_q)
tx_a_data = pack_iq(np.concatenate([pump_i,probe_i]), np.concatenate([pump_q,probe_q]))

i_b,q_b=tone_iq(pi2_samples,0.0,fs=fs_b,phase_deg=0.0)
tx_b_data=pack_iq(np.concatenate([i_b,i_b]),np.concatenate([q_b,q_b]))

assert tx_a_data.size == expected_tx_a_i16
assert tx_b_data.size == expected_tx_b_i16
print('PASS: waveform buffers exactly match programmed TX payload sizes.')


PASS: waveform buffers exactly match programmed TX payload sizes.


In [7]:
# ================= INTERNAL TX TABLES =================
tx_b.mmio.write(TXM_NUM_SEGMENTS,0)
for idx,(active,gap) in enumerate([(pi2_beats,beats_for(PI2_GAP_S)),(pi2_beats,0)]):
    tx_b.mmio.write(TXM_SEG_SEL,idx)
    tx_b.mmio.write(TXM_ACTIVE_LEN,int(active))
    tx_b.mmio.write(TXM_GAP_LEN,int(gap))
    tx_b.mmio.write(TXM_COMMIT,1)
    time.sleep(100e-6)
    if tx_b.mmio.read(TXM_STATUS)&TX_OVERRUN:
        raise RuntimeError(f'DAC-B table commit {idx} overrun')
tx_b.mmio.write(TXM_NUM_SEGMENTS,2)

tx_a.mmio.write(TXM_NUM_SEGMENTS,0)
for idx in range(5):
    tx_a.mmio.write(TXM_SEG_SEL,idx)
    tx_a.mmio.write(TXM_ACTIVE_LEN,int(probe_beats))
    tx_a.mmio.write(TXM_GAP_LEN,int(beats_for(PROBE_GAP_S) if idx<4 else 0))
    tx_a.mmio.write(TXM_COMMIT,1)
    time.sleep(100e-6)
    if tx_a.mmio.read(TXM_STATUS)&TX_OVERRUN:
        raise RuntimeError(f'DAC-A probe table commit {idx} overrun')
tx_a.mmio.write(TXM_NUM_SEGMENTS,5)

assert not (tx_a.mmio.read(TXM_STATUS)&TX_OVERRUN)
assert not (tx_b.mmio.read(TXM_STATUS)&TX_OVERRUN)
print('PASS: DAC-B Pi/2 and DAC-A probe internal tables committed.')


PASS: DAC-B Pi/2 and DAC-A probe internal tables committed.


In [8]:
# ================= DMA + LOOP HELPERS =================
DMASR_HALTED=1<<0; DMASR_IDLE=1<<1; DMASR_ERR_MASK=(1<<4)|(1<<5)|(1<<6)

def dma_status(ch): return int(ch._mmio.read(int(ch._offset)+0x04))
def dma_flags(v):
    out=['halted' if v&1 else 'running']
    if v&2: out.append('idle')
    if v&0x10: out.append('INTERNAL_ERR')
    if v&0x20: out.append('SLAVE_ERR')
    if v&0x40: out.append('DECODE_ERR')
    return '|'.join(out)

def ensure_running(ch,label):
    st=dma_status(ch)
    if st&DMASR_ERR_MASK: raise RuntimeError(f'{label}: DMA error before shot: 0x{st:08x} ({dma_flags(st)})')
    if st&DMASR_HALTED:
        ch.start(); t0=time.perf_counter()
        while time.perf_counter()-t0<0.2:
            st=dma_status(ch)
            if st&DMASR_ERR_MASK: raise RuntimeError(f'{label}: DMA error restarting: 0x{st:08x} ({dma_flags(st)})')
            if not(st&DMASR_HALTED): return
            time.sleep(1e-5)
        raise TimeoutError(f'{label}: DMA did not become running: 0x{st:08x}')

def wait_dma_idle(ch,label,timeout_s):
    t0=time.perf_counter()
    while True:
        st=dma_status(ch)
        if st&DMASR_ERR_MASK: raise RuntimeError(f'{label}: DMA error: 0x{st:08x} ({dma_flags(st)})')
        if st&DMASR_IDLE: return
        if time.perf_counter()-t0>timeout_s: raise TimeoutError(f'{label}: DMA not idle: 0x{st:08x} ({dma_flags(st)})')
        time.sleep(50e-6)

def wait_cap_idle(g,label,timeout_s):
    t0=time.perf_counter()
    while True:
        st=int(g.mmio.read(CAP_STATUS))
        if not(st&CAP_BUSY): return st
        if time.perf_counter()-t0>timeout_s: raise TimeoutError(f'{label}: Capture_Gate busy: 0x{st:08x}')
        time.sleep(100e-6)

def wait_tx_idle(g,label,timeout_s):
    t0=time.perf_counter()
    while True:
        st=int(g.mmio.read(TXM_STATUS))
        if not(st&TX_BUSY): return st
        if time.perf_counter()-t0>timeout_s: raise TimeoutError(f'{label}: TX gate busy: 0x{st:08x}')
        time.sleep(20e-6)

def discard_rx(buf):
    if DISCARD_RX_DATA:
        inv=getattr(buf,'invalidate',None)
        if inv:
            try: inv()
            except Exception: pass

print('Loop helpers ready.')


Loop helpers ready.


In [9]:
# ================= NETWORK / ETHERNET TRANSFER HELPERS =================
# Small length-prefixed TCP protocol, mirrored by RFDC_PC_Ethernet_Receiver.ipynb on the PC:
#   handshake (once):  b'RFDC' + uint32(len) + JSON metadata   -> PC replies b'OK'
#   per shot:          b'SHOT' + uint32(len) + JSON header     -> raw rx_a bytes -> raw rx_b
#                       bytes                                   -> PC replies b'OK'
# Both sides are little-endian int16 raw samples (true for aarch64 PS and any x86_64/ARM PC).
import socket, json, struct

MAGIC_HELLO = b'RFDC'
MAGIC_SHOT  = b'SHOT'
ACK = b'OK'

def _recv_exact(sock, n):
    buf = bytearray(n)
    view = memoryview(buf)
    got = 0
    while got < n:
        r = sock.recv_into(view[got:], n - got)
        if r == 0:
            raise ConnectionError('PC closed the connection mid-transfer')
        got += r
    return bytes(buf)

def eth_connect(host, port, timeout_s):
    s = socket.create_connection((host, port), timeout=timeout_s)
    # Small headers go out far more often than the payload dominates timing on; disabling
    # Nagle avoids adding latency to those without materially affecting bulk throughput.
    s.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
    return s

def _send_json(sock, magic, obj):
    payload = json.dumps(obj).encode('utf-8')
    sock.sendall(magic + struct.pack('>I', len(payload)) + payload)

def eth_handshake(sock, meta):
    _send_json(sock, MAGIC_HELLO, meta)
    ack = _recv_exact(sock, 2)
    if ack != ACK:
        raise RuntimeError(f'PC receiver did not ACK handshake, got {ack!r}')

def eth_send_shot(sock, shot_num, header, rx_a, rx_b):
    hdr = dict(header); hdr['shot'] = shot_num
    _send_json(sock, MAGIC_SHOT, hdr)
    sock.sendall(memoryview(np.asarray(rx_a)))
    sock.sendall(memoryview(np.asarray(rx_b)))
    ack = _recv_exact(sock, 2)
    if ack != ACK:
        raise RuntimeError(f'shot {shot_num}: PC receiver did not ACK, got {ack!r}')

print('Ethernet transfer helpers ready.')


Ethernet transfer helpers ready.


In [10]:
# ================= TTL-TRIGGERED PRODUCTION SEQUENCE (programmed once) =================
# Same 23-row production table as the software-timed notebook (rows 0-21 byte-for-byte
# identical), except row 22's GAP is TTL_GUARD_GAP_S instead of 100 ms -- see the markdown
# at the top of this notebook. TTL_MODE is left off while loading, then set to 1 exactly once
# after the table is committed; it is NOT touched again during the loop (SEQ_RESET, called
# once per shot below, does not clear TTL_MODE -- only the run-state bits and TTL_EDGE_COUNT).

def program_production_sequence():
    row_gap=seq_cycles_for(PUMP_ROW_SPACING_S); START_GAP=8
    pi2_total_s=(2*pi2_beats+beats_for(PI2_GAP_S))/AXIS_BEAT_HZ
    pi2_row_gap=seq_cycles_for(pi2_total_s+5e-6)

    seq.mmio.write(SEQ_TTL_MODE,0)
    seq.mmio.write(SEQ_ENABLE,0); seq.mmio.write(SEQ_RESET,1); time.sleep(100e-6)

    seq.mmio.write(SEQ_ROW_SEL,0); seq.mmio.write(SEQ_MASK,(1<<LANE_TX_A)|(1<<LANE_RX_A)|(1<<LANE_RX_B)); seq.mmio.write(SEQ_GAP,START_GAP)
    seq.mmio.write(SEQ_DUR0,0); seq.mmio.write(SEQ_DUR1,pump_beats); seq.mmio.write(SEQ_DUR2,capture_beats); seq.mmio.write(SEQ_DUR3,capture_beats); seq.mmio.write(SEQ_COMMIT,1)
    for r in range(1,20):
        seq.mmio.write(SEQ_ROW_SEL,r); seq.mmio.write(SEQ_MASK,1<<LANE_TX_A); seq.mmio.write(SEQ_GAP,row_gap)
        seq.mmio.write(SEQ_DUR0,0); seq.mmio.write(SEQ_DUR1,pump_beats); seq.mmio.write(SEQ_DUR2,0); seq.mmio.write(SEQ_DUR3,0); seq.mmio.write(SEQ_COMMIT,1)
    seq.mmio.write(SEQ_ROW_SEL,20); seq.mmio.write(SEQ_MASK,1<<LANE_TX_B); seq.mmio.write(SEQ_GAP,row_gap)
    seq.mmio.write(SEQ_DUR0,0); seq.mmio.write(SEQ_DUR1,0); seq.mmio.write(SEQ_DUR2,0); seq.mmio.write(SEQ_DUR3,0); seq.mmio.write(SEQ_COMMIT,1)
    seq.mmio.write(SEQ_ROW_SEL,21); seq.mmio.write(SEQ_MASK,1<<LANE_TX_A); seq.mmio.write(SEQ_GAP,pi2_row_gap)
    seq.mmio.write(SEQ_DUR0,0); seq.mmio.write(SEQ_DUR1,0); seq.mmio.write(SEQ_DUR2,0); seq.mmio.write(SEQ_DUR3,0); seq.mmio.write(SEQ_COMMIT,1)
    # Guard row: SHORTENED gap vs. the software-timed notebook's 100ms -- see markdown above.
    seq.mmio.write(SEQ_ROW_SEL,22); seq.mmio.write(SEQ_MASK,0); seq.mmio.write(SEQ_GAP,seq_cycles_for(TTL_GUARD_GAP_S))
    seq.mmio.write(SEQ_DUR0,0); seq.mmio.write(SEQ_DUR1,0); seq.mmio.write(SEQ_DUR2,0); seq.mmio.write(SEQ_DUR3,0); seq.mmio.write(SEQ_COMMIT,1)
    seq.mmio.write(SEQ_TABLE_LEN,23); seq.mmio.write(SEQ_CDC_OVERRUN,0xF)
    if int(seq.mmio.read(SEQ_CDC_OVERRUN)&0xF): raise RuntimeError('Sequencer CDC overrun before loop')

    # Enter TTL mode once. Table is loaded, ttl_running guaranteed 0 by the SEQ_RESET above.
    seq.mmio.write(SEQ_TTL_MODE,1)

    probe_start=(START_GAP/SEQ_CLK_HZ)+20*(row_gap/SEQ_CLK_HZ)+(pi2_row_gap/SEQ_CLK_HZ)
    expected_pass=probe_start+TTL_GUARD_GAP_S
    print(f'Production sequence: probe ~{probe_start*1e3:.3f} ms; expected RUN_DONE ~{expected_pass*1e3:.3f} ms after each edge; capture={CAPTURE_S*1e3:.1f} ms')
    if CAPTURE_S <= probe_start+POST_SEQUENCE_MARGIN_S: raise RuntimeError('CAPTURE_S too short for full sequence')
    return probe_start, expected_pass

probe_start_s, expected_pass_s = program_production_sequence()
print('PASS: exact 23-row production schedule programmed, TTL_MODE=1.')


Production sequence: probe ~23.105 ms; expected RUN_DONE ~25.105 ms after each edge; capture=30.0 ms
PASS: exact 23-row production schedule programmed, TTL_MODE=1.


In [11]:
# ================= REUSABLE DMA BUFFERS =================
txbuf_a=base.device.get_memory_by_idx(1).allocate(shape=tx_a_data.shape,dtype=np.int16)
txbuf_b=base.device.get_memory_by_idx(1).allocate(shape=tx_b_data.shape,dtype=np.int16)
rxbuf_a=allocate(shape=(capture_samples,),dtype=np.int16)
rxbuf_b=allocate(shape=(capture_samples,),dtype=np.int16)
txbuf_a[:]=tx_a_data; txbuf_b[:]=tx_b_data; txbuf_a.flush(); txbuf_b.flush()
print('TX A:',hex(int(txbuf_a.physical_address)),txbuf_a.nbytes)
print('TX B:',hex(int(txbuf_b.physical_address)),txbuf_b.nbytes)
print('RX A:',hex(int(rxbuf_a.physical_address)),rxbuf_a.nbytes)
print('RX B:',hex(int(rxbuf_b.physical_address)),rxbuf_b.nbytes)
assert int(txbuf_a.physical_address)>=0x1000000000 and int(txbuf_b.physical_address)>=0x1000000000
assert txbuf_a.nbytes==expected_tx_a_i16*2 and txbuf_b.nbytes==expected_tx_b_i16*2
assert rxbuf_a.nbytes==capture_bytes and rxbuf_b.nbytes==capture_bytes
print('PASS: buffer topology matches the known-good notebook.')


TX A: 0x1000000000 9855040
TX B: 0x1000967000 983040
RX A: 0x78e00000 7372800
RX B: 0x79600000 7372800
PASS: buffer topology matches the known-good notebook.


In [12]:
# ================= CONNECT TO PC + HANDSHAKE =================
# Recompute the same schedule-derived timestamps program_production_sequence() used
# internally (pure function of the USER CONFIG constants above -- no hardware state), so the
# PC can reproduce the LongCapture-style zoom windows without re-deriving sequencer timing.
_row_gap = seq_cycles_for(PUMP_ROW_SPACING_S)
_START_GAP = 8
_pi2_total_s = (2*pi2_beats + beats_for(PI2_GAP_S)) / AXIS_BEAT_HZ
_pi2_row_gap = seq_cycles_for(_pi2_total_s + 5e-6)
_pump_total_s = (_START_GAP + 20*_row_gap) / SEQ_CLK_HZ

print(f'Connecting to PC receiver at {PC_HOST}:{PC_PORT} ...')
eth_sock = eth_connect(PC_HOST, PC_PORT, ETH_TIMEOUT_S)
print('Connected. Sending acquisition metadata...')

handshake_meta = dict(
    AXIS_BEAT_HZ=AXIS_BEAT_HZ, SAMPLES_PER_BEAT=SAMPLES_PER_BEAT, SEQ_CLK_HZ=SEQ_CLK_HZ,
    fs_hz=122.88e6,
    DAC_A_NCO_MHZ=DAC_A_NCO_MHZ, DAC_B_NCO_MHZ=DAC_B_NCO_MHZ,
    PUMP_REPS=PUMP_REPS, PUMP_PULSE_S=PUMP_PULSE_S, PUMP_ROW_SPACING_S=PUMP_ROW_SPACING_S,
    DAC_A_CHIRP_START_MHZ=DAC_A_CHIRP_START_MHZ, DAC_A_CHIRP_END_MHZ=DAC_A_CHIRP_END_MHZ,
    DAC_A_SINE_MHZ=DAC_A_SINE_MHZ,
    PI2_PULSE_S=PI2_PULSE_S, PI2_GAP_S=PI2_GAP_S,
    PROBE_PULSE_S=PROBE_PULSE_S, PROBE_GAP_S=PROBE_GAP_S, DAC_A_PROBE_MHZ=DAC_A_PROBE_MHZ,
    CAPTURE_S=CAPTURE_S, capture_samples=int(capture_samples), capture_bytes=int(capture_bytes),
    LOOP_COUNT=LOOP_COUNT,
    probe_start_s=probe_start_s, expected_pass_s=expected_pass_s,
    pump_total_s=_pump_total_s, pi2_total_s=_pi2_total_s,
)
eth_handshake(eth_sock, handshake_meta)
print('PASS: PC receiver ACKed handshake; per-shot streaming can begin.')


Connecting to PC receiver at 192.168.3.137:5001 ...
Connected. Sending acquisition metadata...
PASS: PC receiver ACKed handshake; per-shot streaming can begin.


In [13]:
# ================= LOOP: TTL-TRIGGERED PRODUCTION PUMP + PI/2 + PROBE =================
def prepare_shot():
    wait_cap_idle(cap_a,'CAP-A',2.0); wait_cap_idle(cap_b,'CAP-B',2.0)
    wait_tx_idle(tx_a,'TX-A',1.0); wait_tx_idle(tx_b,'TX-B',1.0)
    cap_a.mmio.write(CAP_LENGTH,capture_beats); cap_b.mmio.write(CAP_LENGTH,capture_beats)
    cap_a.mmio.write(CAP_CLEAR,CAP_OVERFLOW); cap_b.mmio.write(CAP_CLEAR,CAP_OVERFLOW)
    tx_a.mmio.write(TXM_STATUS,TX_OVERRUN); tx_b.mmio.write(TXM_STATUS,TX_OVERRUN)
    # SEQ_RESET here also zeroes TTL_ARM/RUNNING/RUN_DONE/TTL_EDGE_COUNT in this build (see
    # notebook header markdown) -- that's why run_shot() checks edge_count==1, not a running
    # delta, after each shot. TTL_MODE itself is untouched by this reset and stays at 1.
    seq.mmio.write(SEQ_ENABLE,0); seq.mmio.write(SEQ_RESET,1); time.sleep(100e-6); seq.mmio.write(SEQ_CDC_OVERRUN,0xF)
    seq.mmio.write(SEQ_TTL_STATUS_CLEAR,1)
    for ch,label in [(dma_ra.recvchannel,'RX-A'),(dma_rb.recvchannel,'RX-B'),(dma_ta.sendchannel,'TX-A'),(dma_tb.sendchannel,'TX-B')]: ensure_running(ch,label)

def arm_shot():
    dma_ra.recvchannel.transfer(rxbuf_a); dma_rb.recvchannel.transfer(rxbuf_b)
    dma_ta.sendchannel.transfer(txbuf_a); dma_tb.sendchannel.transfer(txbuf_b)
    time.sleep(DMA_ARM_SETTLE_S)
    for ch,label in [(dma_ra.recvchannel,'RX-A'),(dma_rb.recvchannel,'RX-B'),(dma_ta.sendchannel,'TX-A'),(dma_tb.sendchannel,'TX-B')]:
        st=dma_status(ch)
        if st&(DMASR_ERR_MASK|DMASR_HALTED): raise RuntimeError(f'{label}: bad after arm 0x{st:08x} ({dma_flags(st)})')

# Updated after each shot's hardware is confirmed idle and its data is valid (i.e. right
# before that shot's own Ethernet transfer starts). Used by the *next* call to run_shot() to
# time the full dead time: previous shot done -> this shot armed -> this shot's Ethernet
# transfer ACKed. None for shot 1 (no previous shot to measure from).
last_shot_end_t = None

def run_shot(k):
    global last_shot_end_t
    t0=time.perf_counter(); prepare_shot(); arm_shot()
    t_armed=time.perf_counter()
    gap_before_s = (t_armed - last_shot_end_t) if last_shot_end_t is not None else None

    # Fire: with the sync_out->trig_in jumper installed, arming is the only thing needed --
    # the accepted edge is self-generated the instant TTL_ARM is written.
    ttl_arm(seq)

    deadline=time.perf_counter()+max(2.0, expected_pass_s*10+0.5)
    while True:
        cdc=int(seq.mmio.read(SEQ_CDC_OVERRUN)&0xF); tsa=int(tx_a.mmio.read(TXM_STATUS)); tsb=int(tx_b.mmio.read(TXM_STATUS)); ca=int(cap_a.mmio.read(CAP_STATUS)); cb=int(cap_b.mmio.read(CAP_STATUS))
        if cdc: raise RuntimeError(f'shot {k}: CDC overrun 0x{cdc:x}')
        if tsa&TX_OVERRUN or tsb&TX_OVERRUN: raise RuntimeError(f'shot {k}: TX gate overrun')
        if ca&CAP_OVERFLOW or cb&CAP_OVERFLOW: raise RuntimeError(f'shot {k}: RX overflow')
        if int(seq.mmio.read(SEQ_TTL_STATUS)) & TTL_RUN_DONE: break
        if time.perf_counter()>deadline:
            v=ttl_status_val(seq)
            raise TimeoutError(f'shot {k}: RUN_DONE timeout, STATUS=0x{v:02x} ({ttl_status_str(v)}), row_ptr={int(seq.mmio.read(SEQ_ROW_PTR))}')
        time.sleep(0.5e-3)

    ec=ttl_edge_count(seq)
    row=int(seq.mmio.read(SEQ_ROW_PTR))
    if ec != 1: raise RuntimeError(f'shot {k}: expected TTL_EDGE_COUNT==1 (fresh reset baseline), got {ec}')
    if row != 0: raise RuntimeError(f'shot {k}: expected ROW_PTR wrapped to 0, got {row}')
    seq.mmio.write(SEQ_TTL_STATUS_CLEAR,1)

    tout=max(2.0,CAPTURE_S+1.0)
    wait_cap_idle(cap_a,'CAP-A',tout); wait_cap_idle(cap_b,'CAP-B',tout)
    wait_dma_idle(dma_ra.recvchannel,'RX-A',tout); wait_dma_idle(dma_rb.recvchannel,'RX-B',tout)
    wait_dma_idle(dma_ta.sendchannel,'TX-A',2.0); wait_dma_idle(dma_tb.sendchannel,'TX-B',2.0)
    dma_ra.recvchannel.wait(); dma_rb.recvchannel.wait(); dma_ta.sendchannel.wait(); dma_tb.sendchannel.wait()
    rx_a_bytes=int(dma_ra.recvchannel.transferred); rx_b_bytes=int(dma_rb.recvchannel.transferred); tx_a_bytes=int(dma_ta.sendchannel.transferred); tx_b_bytes=int(dma_tb.sendchannel.transferred)
    if rx_a_bytes!=capture_bytes or rx_b_bytes!=capture_bytes: raise RuntimeError(f'shot {k}: RX bytes {rx_a_bytes}/{rx_b_bytes}, expected {capture_bytes}')
    if tx_a_bytes!=txbuf_a.nbytes or tx_b_bytes!=txbuf_b.nbytes: raise RuntimeError(f'shot {k}: TX bytes {tx_a_bytes}/{tx_b_bytes}, expected {txbuf_a.nbytes}/{txbuf_b.nbytes}')
    # invalidate() here is a CPU cache-coherency step, not a data-discard: it makes sure the
    # CPU (and the socket send below, which reads straight out of these buffers) sees the
    # samples the DMA engine actually wrote, not stale cache lines.
    discard_rx(rxbuf_a); discard_rx(rxbuf_b)
    ca=int(cap_a.mmio.read(CAP_STATUS)); cb=int(cap_b.mmio.read(CAP_STATUS)); tsa=int(tx_a.mmio.read(TXM_STATUS)); tsb=int(tx_b.mmio.read(TXM_STATUS)); cdc=int(seq.mmio.read(SEQ_CDC_OVERRUN)&0xF)
    statuses={n:dma_status(ch) for n,ch in [('RX-A',dma_ra.recvchannel),('RX-B',dma_rb.recvchannel),('TX-A',dma_ta.sendchannel),('TX-B',dma_tb.sendchannel)]}
    if ca&CAP_OVERFLOW or cb&CAP_OVERFLOW: raise RuntimeError(f'shot {k}: post-shot overflow')
    if tsa&TX_OVERRUN or tsb&TX_OVERRUN: raise RuntimeError(f'shot {k}: post-shot TX overrun')
    if cdc: raise RuntimeError(f'shot {k}: post-shot CDC overrun')
    for n,st in statuses.items():
        if st&(DMASR_ERR_MASK|DMASR_HALTED): raise RuntimeError(f'shot {k}: {n} unhealthy post-shot 0x{st:08x} ({dma_flags(st)})')

    # Shot's hardware activity is now fully complete and rxbuf_a/rxbuf_b hold valid data.
    shot_end_t = time.perf_counter()
    last_shot_end_t = shot_end_t

    # Ethernet transfer to the PC. Blocking on the PC's ACK (not just the local socket
    # buffer accepting the bytes) so eth_elapsed_s below reflects the true round trip: send +
    # PC-side write-to-disk + ACK. Nothing else touches rxbuf_a/rxbuf_b until this returns, so
    # there's no race with the next shot's DMA re-arming it.
    eth_t0=time.perf_counter()
    eth_send_shot(eth_sock, k, {'row':row,'edge_count':ec,'cdc':cdc,
                                'rx_a_bytes':rx_a_bytes,'rx_b_bytes':rx_b_bytes}, rxbuf_a, rxbuf_b)
    eth_elapsed_s=time.perf_counter()-eth_t0

    return {'shot':k,'row':row,'edge_count':ec,'rx_a':rx_a_bytes,'rx_b':rx_b_bytes,'tx_a':tx_a_bytes,'tx_b':tx_b_bytes,'cap_a':ca,'cap_b':cb,'cdc':cdc,'dma':statuses,
            'elapsed_s':time.perf_counter()-t0,'gap_before_s':gap_before_s,'eth_elapsed_s':eth_elapsed_s}

results=[]
print(f'RUNNING {LOOP_COUNT} identical TTL-triggered production shots; RX data streamed to {PC_HOST}:{PC_PORT} over Ethernet after each shot.')
for k in range(1,LOOP_COUNT+1):
    try:
        r=run_shot(k); results.append(r)
        gap_str = f"{r['gap_before_s']*1e3:.1f}ms" if r['gap_before_s'] is not None else 'n/a (first shot)'
        print(f"shot {k:04d}: row={r['row']} ec={r['edge_count']} RX={r['rx_a']}/{r['rx_b']} TX={r['tx_a']}/{r['tx_b']} ovf={bool(r['cap_a']&CAP_OVERFLOW) or bool(r['cap_b']&CAP_OVERFLOW)} cdc={r['cdc']} DMA={'/'.join(hex(v) for v in r['dma'].values())} eth={r['eth_elapsed_s']*1e3:.1f}ms gap_before={gap_str} elapsed={r['elapsed_s']:.3f}s")
    except Exception:
        try:
            seq.mmio.write(SEQ_ENABLE,0)
        except Exception:
            pass
        if FAIL_FAST: raise
    if INTER_SHOT_PAUSE_S: time.sleep(INTER_SHOT_PAUSE_S)
assert len(results)==LOOP_COUNT
print(f'PASS: {LOOP_COUNT} repeated TTL-triggered production pump + Pi/2 + probe shots, all streamed to the PC.')


RUNNING 2000 identical TTL-triggered production shots; RX data streamed to 192.168.3.137:5001 over Ethernet after each shot.
shot 0001: row=0 ec=1 RX=7372800/7372800 TX=9855040/983040 ovf=False cdc=0 DMA=0x1002/0x1002/0x1002/0x1002 eth=499.4ms gap_before=n/a (first shot) elapsed=0.538s
shot 0002: row=0 ec=1 RX=7372800/7372800 TX=9855040/983040 ovf=False cdc=0 DMA=0x1002/0x1002/0x1002/0x1002 eth=516.7ms gap_before=507.9ms elapsed=0.554s
shot 0003: row=0 ec=1 RX=7372800/7372800 TX=9855040/983040 ovf=False cdc=0 DMA=0x1002/0x1002/0x1002/0x1002 eth=516.7ms gap_before=525.2ms elapsed=0.554s
shot 0004: row=0 ec=1 RX=7372800/7372800 TX=9855040/983040 ovf=False cdc=0 DMA=0x1002/0x1002/0x1002/0x1002 eth=512.8ms gap_before=525.2ms elapsed=0.550s
shot 0005: row=0 ec=1 RX=7372800/7372800 TX=9855040/983040 ovf=False cdc=0 DMA=0x1002/0x1002/0x1002/0x1002 eth=472.8ms gap_before=521.3ms elapsed=0.510s
shot 0006: row=0 ec=1 RX=7372800/7372800 TX=9855040/983040 ovf=False cdc=0 DMA=0x1002/0x1002/0x1002/0


KeyboardInterrupt



In [ ]:
# ================= SUMMARY + TIMING STATS + CLEANUP =================
print('shot,elapsed_s,eth_elapsed_s,gap_before_s,rx_a,rx_b,tx_a,tx_b,cap_a,cap_b,cdc,edge_count')
for r in results:
    gap=r['gap_before_s']
    print(r['shot'], f"{r['elapsed_s']:.4f}", f"{r['eth_elapsed_s']:.4f}",
          ('n/a' if gap is None else f"{gap:.4f}"),
          r['rx_a'],r['rx_b'],r['tx_a'],r['tx_b'],r['cap_a'],r['cap_b'],r['cdc'],r['edge_count'],sep=',')

gaps=[r['gap_before_s'] for r in results if r['gap_before_s'] is not None]
eth_times=[r['eth_elapsed_s'] for r in results]
print()
if gaps:
    max_gap=max(gaps); max_gap_shot=results[gaps.index(max_gap)+1]['shot']
    print(f'Max inter-shot dead time (prior shot HW done -> this shot armed & Ethernet ACKed): '
          f'{max_gap*1e3:.1f} ms (shot {max_gap_shot})')
    print(f'Mean inter-shot dead time: {sum(gaps)/len(gaps)*1e3:.1f} ms')
    print(f'Min inter-shot dead time:  {min(gaps)*1e3:.1f} ms')
print(f'Max Ethernet transfer time (send + PC write + ACK): {max(eth_times)*1e3:.1f} ms')
print(f'Mean Ethernet transfer time: {sum(eth_times)/len(eth_times)*1e3:.1f} ms')
if gaps:
    print()
    print(f'This max ({max_gap*1e3:.1f} ms) is the minimum safe period between external trigger')
    print(f'edges for this configuration/network/PC -- a DG4000 (or anything else) firing faster')
    print(f'than that will land while this board is still mid-transfer or mid-rearm for the')
    print(f'previous shot. It already includes prepare_shot()+arm_shot() time, so the separate')
    print(f'INTER_SHOT_PAUSE_S ({INTER_SHOT_PAUSE_S*1e3:.1f} ms) sleep below is now a minor add-on,')
    print(f'not the dominant term -- the Ethernet transfer is.')

print('RX data were streamed to the PC over Ethernet after every shot; TX data were reused from PL DDR4 for every shot.')

seq.mmio.write(SEQ_TTL_MODE,0)
seq.mmio.write(SEQ_ENABLE,0)

try:
    eth_sock.shutdown(socket.SHUT_RDWR)
except Exception:
    pass
eth_sock.close()
print('Ethernet connection to PC closed.')

# Uncomment when the full loop is complete and no more shots are desired.
# for obj in (rxbuf_a,rxbuf_b,txbuf_a,txbuf_b):
#     try: obj.freebuffer()
#     except Exception: pass


## Next step once this passes

Remove the `sync_out`<->`trig_in` jumper and wire in the level-shifted DG4000 for the real
trigger source; re-run unchanged. As with the single-shot notebook, this confirms the RTL and
gate/DMA path under repeated hardware-triggered starts, but not the DG4000's own signal
levels/timing or the eventual board<->DG4000 mutual-triggering loop -- those are separate,
unverified next steps.

## On the 0.5 s timeout, Ethernet vs USB, and required dead time

The `+0.5` in `run_shot()`'s `deadline=time.perf_counter()+max(2.0, expected_pass_s*10+0.5)` is
a timeout for detecting `RUN_DONE` on *this* shot's own sequencer pass -- it has nothing to do
with Ethernet and doesn't need to change: the transfer only starts after that wait already
succeeded. The number that actually matters for back-to-back shots is the measured
**max inter-shot dead time** printed in the summary cell above; use that (with margin) as the
minimum period for any future external trigger, not the 0.5 s constant.

At this notebook's default 30 ms `CAPTURE_S`, each shot's two RX channels total ~14 MiB. At a
realistic 180-200 Mbps on this board's RJ45 (its DAC/ADC-side fabric clocking currently caps
the PS GEM link there), that's roughly 0.6-0.75 s of transfer time per shot, not the ~2 s
originally guessed -- see the printed per-shot `eth_elapsed_s` above for the real number on
your network. If the RFSoC's Ethernet clocking gets fixed to reach ~950 Mbps, the same payload
drops to roughly 125-150 ms.

If USB3 to an external SSD off one of the board's USB-A ports turns out faster in practice,
the same `eth_send_shot`-shaped pipeline (open a file instead of a socket, write bytes, no ACK
round trip needed) drops in with only the transport swapped -- ask if you want that variant.
